In [6]:
# =========================================================
# 1. 라이브러리 불러오기
# =========================================================

import os
import time
import json

from pathlib import Path

import pandas as pd
import requests

from dotenv import load_dotenv
from tqdm.auto import tqdm

In [7]:
# =========================================================
# 2. 프로젝트 경로 설정
# =========================================================

CURRENT_DIR = Path.cwd()

# 노트북은 GitHub 저장소의 notebooks 폴더 안에 있음
if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
else:
    REPO_DIR = CURRENT_DIR

# GitHub 저장소와 project_data는 같은 상위 폴더에 있음
WORKSPACE_DIR = REPO_DIR.parent

DATA_DIR = WORKSPACE_DIR / "project_data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
API_TEST_DIR = DATA_DIR / "api_test"

ENV_FILE = DATA_DIR / ".env"

ADM_OLD_FILE = (
    RAW_DIR
    / "seoul_admi_list_2023_to_202507.csv"
)

ADM_NEW_FILE = (
    RAW_DIR
    / "seoul_admi_list_202508_to_202606.csv"
)

API_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("현재 작업 위치:", CURRENT_DIR)
print("GitHub 프로젝트:", REPO_DIR)
print("데이터 폴더:", DATA_DIR)
print("환경변수 파일:", ENV_FILE)
print("구 행정동 코드표:", ADM_OLD_FILE)
print("신 행정동 코드표:", ADM_NEW_FILE)

print("\n[파일 존재 확인]")
print(".env:", ENV_FILE.exists())
print("구 코드표:", ADM_OLD_FILE.exists())
print("신 코드표:", ADM_NEW_FILE.exists())

현재 작업 위치: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
GitHub 프로젝트: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project
데이터 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data
환경변수 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/.env
구 행정동 코드표: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw/seoul_admi_list_2023_to_202507.csv
신 행정동 코드표: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw/seoul_admi_list_202508_to_202606.csv

[파일 존재 확인]
.env: True
구 코드표: True
신 코드표: True


In [9]:
# =========================================================
# 3. 환경변수 불러오기
# =========================================================

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f".env 파일을 찾을 수 없습니다.\n"
        f"확인 경로: {ENV_FILE}"
    )

load_dotenv(ENV_FILE)

KAKAO_REST_API_KEY = os.getenv(
    "KAKAO_REST_API_KEY"
)

TMAP_APP_KEY = os.getenv(
    "TMAP_APP_KEY"
)

missing_keys = []

if not KAKAO_REST_API_KEY:
    missing_keys.append(
        "KAKAO_REST_API_KEY"
    )

if not TMAP_APP_KEY:
    missing_keys.append(
        "TMAP_APP_KEY"
    )

if missing_keys:
    raise ValueError(
        ".env 파일에 다음 API 키가 없습니다: "
        + ", ".join(missing_keys)
    )

print(
    "카카오 REST API 키 로드:",
    bool(KAKAO_REST_API_KEY),
)

print(
    "TMAP APP KEY 로드:",
    bool(TMAP_APP_KEY),
)

카카오 REST API 키 로드: True
TMAP APP KEY 로드: True


In [10]:
# =========================================================
# 4. 07번 결과 파일 불러오기
# =========================================================

GEOCODE_TARGET_FILE = (
    PROCESSED_DIR
    / "commute_dong_geocode_targets.csv"
)

API_TARGET_FILE = (
    PROCESSED_DIR
    / "commute_api_targets_80.csv"
)

if not GEOCODE_TARGET_FILE.exists():
    raise FileNotFoundError(
        f"좌표 검색 대상 파일이 없습니다.\n"
        f"확인 경로: {GEOCODE_TARGET_FILE}"
    )

if not API_TARGET_FILE.exists():
    raise FileNotFoundError(
        f"API 호출 대상 파일이 없습니다.\n"
        f"확인 경로: {API_TARGET_FILE}"
    )

geocode_targets = pd.read_csv(
    GEOCODE_TARGET_FILE,
    encoding="utf-8-sig",
    dtype={
        "행정동 코드": "string",
        "행정동 이름": "string",
    },
)

api_targets = pd.read_csv(
    API_TARGET_FILE,
    encoding="utf-8-sig",
    dtype={
        "OD_ID": "string",
        "거주동 코드": "string",
        "거주동 이름": "string",
        "근무동 코드": "string",
        "근무동 이름": "string",
    },
)

print("좌표 검색 대상 행정동:", len(geocode_targets))
print("경로 API 대상 OD:", len(api_targets))

display(geocode_targets.head())

좌표 검색 대상 행정동: 428
경로 API 대상 OD: 30411


,행정동 코드,행정동 이름,주민센터_검색어,대표_장소명,대표_주소,대표_위도,대표_경도,지오코딩_API,지오코딩_상태,지오코딩_오류
0,11110515,청운효자동,서울특별시 청운효자동 주민센터,NaN,NaN,NaN,NaN,NaN,미호출,NaN
1,11110530,사직동,서울특별시 사직동 주민센터,NaN,NaN,NaN,NaN,NaN,미호출,NaN
2,11110540,삼청동,서울특별시 삼청동 주민센터,NaN,NaN,NaN,NaN,NaN,미호출,NaN
3,11110550,부암동,서울특별시 부암동 주민센터,NaN,NaN,NaN,NaN,NaN,미호출,NaN
4,11110560,평창동,서울특별시 평창동 주민센터,NaN,NaN,NaN,NaN,NaN,미호출,NaN


In [7]:
print(type(geocode_targets))
print(geocode_targets.shape)
print(geocode_targets.columns.tolist())

<class 'pandas.DataFrame'>
(428, 10)
['행정동 코드', '행정동 이름', '주민센터_검색어', '대표_장소명', '대표_주소', '대표_위도', '대표_경도', '지오코딩_API', '지오코딩_상태', '지오코딩_오류']


In [4]:
# =========================================================
# 5. 행정동 코드표 불러오기 및 자치구 결합
# =========================================================

adm_old = pd.read_csv(
    ADM_OLD_FILE,
    encoding="utf-8-sig",
    dtype="string",
)

adm_new = pd.read_csv(
    ADM_NEW_FILE,
    encoding="utf-8-sig",
    dtype="string",
)

print("구 코드표 컬럼:", adm_old.columns.tolist())
print("신 코드표 컬럼:", adm_new.columns.tolist())

required_adm_cols = [
    "자치구",
    "행정동코드",
    "행정동명",
]

for table_name, table in {
    "구 코드표": adm_old,
    "신 코드표": adm_new,
}.items():
    missing = [
        col
        for col in required_adm_cols
        if col not in table.columns
    ]

    if missing:
        raise ValueError(
            f"{table_name}에 필요한 컬럼이 없습니다: {missing}"
        )

# 구·신 코드표 통합
adm_ref = (
    pd.concat(
        [
            adm_old[required_adm_cols],
            adm_new[required_adm_cols],
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["행정동코드"],
        keep="last",
    )
    .rename(
        columns={
            "행정동코드": "행정동 코드",
            "행정동명": "코드표 행정동 이름",
        }
    )
)

# 문자열 정리
adm_ref["행정동 코드"] = (
    adm_ref["행정동 코드"]
    .astype("string")
    .str.strip()
)

geocode_targets["행정동 코드"] = (
    geocode_targets["행정동 코드"]
    .astype("string")
    .str.strip()
)

# 재실행 대비 기존 결합 컬럼 제거
geocode_targets = geocode_targets.drop(
    columns=[
        "자치구",
        "코드표 행정동 이름",
    ],
    errors="ignore",
)

# 자치구 결합
geocode_targets = geocode_targets.merge(
    adm_ref,
    on="행정동 코드",
    how="left",
    validate="one_to_one",
)

print(
    "자치구 미결합 건수:",
    geocode_targets["자치구"].isna().sum(),
)

display(
    geocode_targets[
        [
            "행정동 코드",
            "자치구",
            "행정동 이름",
            "코드표 행정동 이름",
        ]
    ].head(20)
)

구 코드표 컬럼: ['자치구', '행정동코드', '행정동명']
신 코드표 컬럼: ['자치구', '행정동코드', '행정동명']


NameError: name 'geocode_targets' is not defined

In [10]:
# =========================================================
# 6. 자치구를 포함한 주민센터 검색어 재생성
# =========================================================

if geocode_targets["자치구"].isna().any():
    raise ValueError(
        "자치구가 결합되지 않은 행이 있어 검색어를 만들 수 없습니다."
    )

geocode_targets["주민센터_검색어"] = (
    "서울특별시 "
    + geocode_targets["자치구"].str.strip()
    + " "
    + geocode_targets["행정동 이름"].str.strip()
    + " 주민센터"
)

print("검색어 생성 완료:", len(geocode_targets))

# 이름이 중복되는 행정동 확인
duplicate_names = geocode_targets[
    geocode_targets["행정동 이름"].duplicated(
        keep=False
    )
].sort_values(
    [
        "행정동 이름",
        "자치구",
    ]
)

display(
    duplicate_names[
        [
            "행정동 코드",
            "자치구",
            "행정동 이름",
            "주민센터_검색어",
        ]
    ]
)

검색어 생성 완료: 428


,행정동 코드,자치구,행정동 이름,주민센터_검색어
360,11680510,강남구,신사동,서울특별시 강남구 신사동 주민센터
334,11620685,관악구,신사동,서울특별시 관악구 신사동 주민센터


In [11]:
# =========================================================
# 7. 좌표 검색 대상 데이터 점검
# =========================================================

required_geocode_cols = [
    "행정동 코드",
    "자치구",
    "행정동 이름",
    "주민센터_검색어",
]

missing_cols = [
    col
    for col in required_geocode_cols
    if col not in geocode_targets.columns
]

if missing_cols:
    raise ValueError(
        f"좌표 검색 대상 데이터에 필요한 컬럼이 없습니다: "
        f"{missing_cols}"
    )

duplicate_code_count = (
    geocode_targets["행정동 코드"]
    .duplicated()
    .sum()
)

missing_gu_count = (
    geocode_targets["자치구"]
    .isna()
    .sum()
)

missing_query_count = (
    geocode_targets["주민센터_검색어"]
    .isna()
    .sum()
)

print("행정동 수:", len(geocode_targets))
print("행정동 코드 중복:", duplicate_code_count)
print("자치구 누락:", missing_gu_count)
print("검색어 누락:", missing_query_count)

if duplicate_code_count > 0:
    raise ValueError("행정동 코드 중복이 있습니다.")

if missing_gu_count > 0:
    raise ValueError("자치구 누락이 있습니다.")

if missing_query_count > 0:
    raise ValueError("주민센터 검색어 누락이 있습니다.")

display(
    geocode_targets[
        [
            "행정동 코드",
            "자치구",
            "행정동 이름",
            "주민센터_검색어",
        ]
    ].head(20)
)

행정동 수: 428
행정동 코드 중복: 0
자치구 누락: 0
검색어 누락: 0


,행정동 코드,자치구,행정동 이름,주민센터_검색어
0,11110515,종로구,청운효자동,서울특별시 종로구 청운효자동 주민센터
1,11110530,종로구,사직동,서울특별시 종로구 사직동 주민센터
2,11110540,종로구,삼청동,서울특별시 종로구 삼청동 주민센터
3,11110550,종로구,부암동,서울특별시 종로구 부암동 주민센터
4,11110560,종로구,평창동,서울특별시 종로구 평창동 주민센터
5,11110570,종로구,무악동,서울특별시 종로구 무악동 주민센터
6,11110580,종로구,교남동,서울특별시 종로구 교남동 주민센터
7,11110600,종로구,가회동,서울특별시 종로구 가회동 주민센터
8,11110615,종로구,종로1.2.3.4가동,서울특별시 종로구 종로1.2.3.4가동 주민센터
9,11110630,종로구,종로5.6가동,서울특별시 종로구 종로5.6가동 주민센터


In [12]:
# =========================================================
# 8. 카카오 주민센터 검색 함수
# =========================================================

KAKAO_KEYWORD_URL = (
    "https://dapi.kakao.com"
    "/v2/local/search/keyword.json"
)


def search_kakao_place(
    query: str,
    api_key: str,
    timeout: int = 10,
) -> dict:
    """
    카카오 키워드 장소 검색 API로 장소를 검색한다.

    반환값
    -------
    {
        "status": "성공" | "검색결과없음" | "호출오류",
        "query": 검색어,
        "documents": 검색 결과 목록,
        "error": 오류 메시지
    }
    """

    headers = {
        "Authorization": f"KakaoAK {api_key}",
    }

    params = {
        "query": query,
        "page": 1,
        "size": 15,
        "sort": "accuracy",
    }

    try:
        response = requests.get(
            KAKAO_KEYWORD_URL,
            headers=headers,
            params=params,
            timeout=timeout,
        )

        response.raise_for_status()

        result = response.json()

        documents = result.get(
            "documents",
            [],
        )

        if not documents:
            return {
                "status": "검색결과없음",
                "query": query,
                "documents": [],
                "error": None,
            }

        return {
            "status": "성공",
            "query": query,
            "documents": documents,
            "error": None,
        }

    except requests.RequestException as error:
        return {
            "status": "호출오류",
            "query": query,
            "documents": [],
            "error": str(error),
        }

    except ValueError as error:
        return {
            "status": "응답해석오류",
            "query": query,
            "documents": [],
            "error": str(error),
        }

In [15]:
# =========================================================
# 9. 카카오 주민센터 검색 1건 테스트
# =========================================================

test_candidates = geocode_targets[
    (geocode_targets["자치구"] == "강남구")
    & (geocode_targets["행정동 이름"] == "신사동")
]

if len(test_candidates) != 1:
    raise ValueError(
        f"강남구 신사동 검색 대상이 1건이 아닙니다: "
        f"{len(test_candidates)}건"
    )

test_row = test_candidates.iloc[0]
test_query = test_row["주민센터_검색어"]

print("시험 검색어:", test_query)

test_result = search_kakao_place(
    query=test_query,
    api_key=KAKAO_REST_API_KEY,
)

print("호출 상태:", test_result["status"])
print("검색 결과 수:", len(test_result["documents"]))

if test_result["error"]:
    print("오류:", test_result["error"])

시험 검색어: 서울특별시 강남구 신사동 주민센터
호출 상태: 성공
검색 결과 수: 4


In [16]:
test_documents = pd.DataFrame(test_result["documents"])

display(
    test_documents[
        [
            "place_name",
            "category_name",
            "address_name",
            "road_address_name",
            "x",
            "y",
        ]
    ]
)

,place_name,category_name,address_name,road_address_name,x,y
0,신사동주민센터,"사회,공공기관 > 지방행정기관 > 행정복지센터 > 동행정복지센터",서울 강남구 신사동 548-1,서울 강남구 압구정로 128,127.02277448907,37.523985029861
1,무인민원발급창구 신사동주민센터2,"사회,공공기관 > 지방행정기관 > 무인민원발급창구",서울 강남구 신사동 548-1,서울 강남구 압구정로 128,127.022797110942,37.5239742134674
2,무인민원발급창구 신사동주민센터,"사회,공공기관 > 지방행정기관 > 무인민원발급창구",서울 강남구 신사동 548-1,서울 강남구 압구정로 128,127.022797110942,37.5239742134674
3,강남구 신사동주민센터 전기차충전소,"교통,수송 > 자동차 > 전기차 충전소",서울 강남구 신사동 548-1,서울 강남구 압구정로 128,127.0227371490772,37.52395710597295


In [17]:
def select_best_resident_center(documents):
    """
    카카오 검색 결과 중 실제 동 주민센터를 선택한다.
    """

    if not documents:
        return None

    docs = pd.DataFrame(documents).copy()

    # 실제 동 행정복지센터 카테고리만 우선 선택
    center_docs = docs[
        docs["category_name"]
        .fillna("")
        .str.contains(
            "동행정복지센터",
            regex=False,
        )
    ].copy()

    # 카테고리가 정확히 잡힌 결과가 있으면 첫 번째 사용
    if not center_docs.empty:
        return center_docs.iloc[0].to_dict()

    # 예외적으로 카테고리가 다르게 등록된 경우 명칭으로 보조 검색
    fallback_docs = docs[
        docs["place_name"]
        .fillna("")
        .str.contains(
            "주민센터|행정복지센터",
            regex=True,
        )
        &
        ~docs["place_name"]
        .fillna("")
        .str.contains(
            "주차장|화장실|북카페|도서관|어린이집",
            regex=True,
        )
    ].copy()

    if not fallback_docs.empty:
        return fallback_docs.iloc[0].to_dict()

    return None

In [18]:
best_place = select_best_resident_center(
    test_result["documents"]
)

best_place

{'address_name': '서울 강남구 신사동 548-1',
 'category_group_code': 'PO3',
 'category_group_name': '공공기관',
 'category_name': '사회,공공기관 > 지방행정기관 > 행정복지센터 > 동행정복지센터',
 'distance': '',
 'id': '7967833',
 'phone': '02-3423-7320',
 'place_name': '신사동주민센터',
 'place_url': 'http://place.map.kakao.com/7967833',
 'road_address_name': '서울 강남구 압구정로 128',
 'x': '127.02277448907',
 'y': '37.523985029861'}

In [20]:
# =========================================================
# 10. 전체 주민센터 좌표 수집
# =========================================================

geocode_results = []

for _, row in tqdm(
    geocode_targets.iterrows(),
    total=len(geocode_targets),
    desc="주민센터 좌표 수집",
):
    query = row["주민센터_검색어"]

    result = search_kakao_place(
        query=query,
        api_key=KAKAO_REST_API_KEY,
    )

    best_place = select_best_resident_center(
        result["documents"]
    )

    common_data = {
        "행정동 코드": row["행정동 코드"],
        "자치구": row["자치구"],
        "행정동 이름": row["행정동 이름"],
        "주민센터_검색어": query,
        "검색결과수": len(result["documents"]),
        "지오코딩_API": "Kakao Local",
    }

    if best_place is None:
        geocode_results.append(
            {
                **common_data,
                "대표_장소명": None,
                "대표_주소": None,
                "대표_경도": None,
                "대표_위도": None,
                "지오코딩_상태": result["status"],
                "지오코딩_오류": result["error"],
            }
        )

    else:
        geocode_results.append(
            {
                **common_data,
                "대표_장소명": best_place.get(
                    "place_name"
                ),
                "대표_주소": (
                    best_place.get("road_address_name")
                    or best_place.get("address_name")
                ),
                "대표_경도": float(
                    best_place["x"]
                ),
                "대표_위도": float(
                    best_place["y"]
                ),
                "지오코딩_상태": "성공",
                "지오코딩_오류": None,
            }
        )

    time.sleep(0.05)

주민센터 좌표 수집:   0%|          | 0/428 [00:00<?, ?it/s]

In [21]:
# =========================================================
# 11. 좌표 수집 결과 데이터프레임 생성
# =========================================================

dong_geocoded = pd.DataFrame(
    geocode_results
)

print(
    "전체 행정동 수:",
    f"{len(dong_geocoded):,}",
)

print(
    "좌표 수집 성공:",
    f"{(dong_geocoded['지오코딩_상태'] == '성공').sum():,}",
)

print(
    "좌표 수집 실패:",
    f"{(dong_geocoded['지오코딩_상태'] != '성공').sum():,}",
)

display(
    dong_geocoded.head()
)

전체 행정동 수: 428
좌표 수집 성공: 428
좌표 수집 실패: 0


,행정동 코드,자치구,행정동 이름,주민센터_검색어,검색결과수,지오코딩_API,대표_장소명,대표_주소,대표_경도,대표_위도,지오코딩_상태,지오코딩_오류
0,11110515,종로구,청운효자동,서울특별시 종로구 청운효자동 주민센터,4,Kakao Local,청운효자동주민센터,서울 종로구 자하문로 92,126.970650,37.584116,성공,None
1,11110530,종로구,사직동,서울특별시 종로구 사직동 주민센터,2,Kakao Local,사직동주민센터,서울 종로구 경희궁1길 15,126.970591,37.571354,성공,None
2,11110540,종로구,삼청동,서울특별시 종로구 삼청동 주민센터,2,Kakao Local,삼청동주민센터,서울 종로구 삼청로 107,126.981758,37.584998,성공,None
3,11110550,종로구,부암동,서울특별시 종로구 부암동 주민센터,4,Kakao Local,부암동주민센터,서울 종로구 창의문로 145,126.964061,37.592414,성공,None
4,11110560,종로구,평창동,서울특별시 종로구 평창동 주민센터,4,Kakao Local,평창동주민센터,서울 종로구 평창문화로 65,126.968358,37.606392,성공,None


In [22]:
# =========================================================
# 15. 좌표 결과 점검
# =========================================================

failed = dong_geocoded[
    dong_geocoded["지오코딩_상태"] != "성공"
].copy()

non_seoul = dong_geocoded[
    ~dong_geocoded["대표_주소"]
    .fillna("")
    .str.startswith("서울")
].copy()

# 검색 대상 자치구가 실제 주소에 포함되는지 확인
gu_mismatch = dong_geocoded[
    dong_geocoded.apply(
        lambda row: (
            row["지오코딩_상태"] == "성공"
            and row["자치구"] not in str(row["대표_주소"])
        ),
        axis=1,
    )
].copy()

duplicate_coords = dong_geocoded[
    dong_geocoded.duplicated(
        subset=[
            "대표_경도",
            "대표_위도",
        ],
        keep=False,
    )
].copy()

print("좌표 수집 실패:", len(failed))
print("서울 외 주소:", len(non_seoul))
print("자치구 불일치:", len(gu_mismatch))
print("좌표 중복 행:", len(duplicate_coords))

print("\n[실패]")
display(failed)

print("\n[자치구 불일치]")
display(
    gu_mismatch[
        [
            "행정동 코드",
            "자치구",
            "행정동 이름",
            "대표_장소명",
            "대표_주소",
        ]
    ]
)

print("\n[좌표 중복]")
display(
    duplicate_coords.sort_values(
        [
            "대표_경도",
            "대표_위도",
        ]
    )
)

좌표 수집 실패: 0
서울 외 주소: 0
자치구 불일치: 0
좌표 중복 행: 2

[실패]


,행정동 코드,자치구,행정동 이름,주민센터_검색어,검색결과수,지오코딩_API,대표_장소명,대표_주소,대표_경도,대표_위도,지오코딩_상태,지오코딩_오류



[자치구 불일치]


,행정동 코드,자치구,행정동 이름,대표_장소명,대표_주소



[좌표 중복]


,행정동 코드,자치구,행정동 이름,주민센터_검색어,검색결과수,지오코딩_API,대표_장소명,대표_주소,대표_경도,대표_위도,지오코딩_상태,지오코딩_오류
81,11230533,동대문구,용두동,서울특별시 동대문구 용두동 주민센터,2,Kakao Local,용두동주민센터,서울 동대문구 천호대로27길 35,127.037262,37.575803,성공,None
82,11230536,동대문구,용신동,서울특별시 동대문구 용신동 주민센터,1,Kakao Local,용두동주민센터,서울 동대문구 천호대로27길 35,127.037262,37.575803,성공,None


In [23]:
# =========================================================
# 16. 주민센터 좌표 결과 저장
# =========================================================

DONG_GEOCODED_FILE = (
    PROCESSED_DIR
    / "commute_dong_geocoded_kakao.csv"
)

dong_geocoded.to_csv(
    DONG_GEOCODED_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", DONG_GEOCODED_FILE)
print("저장 행 수:", len(dong_geocoded))

저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_dong_geocoded_kakao.csv
저장 행 수: 428


# 경로 조회 #

In [24]:
# =========================================================
# 14. TMAP 앱키 여러 개 불러오기
# =========================================================

# .env를 수정한 뒤에는 override=True로 다시 불러와야
# 현재 노트북 세션에도 수정 내용이 반영된다.
load_dotenv(
    ENV_FILE,
    override=True,
)

TMAP_APP_KEYS = [
    os.getenv("TMAP_APP_KEY_1"),
    os.getenv("TMAP_APP_KEY_2"),
    os.getenv("TMAP_APP_KEY_3"),
    os.getenv("TMAP_APP_KEY_4"),
    os.getenv("TMAP_APP_KEY_5"),
]

# 비어 있는 키 제거
TMAP_APP_KEYS = [
    key.strip()
    for key in TMAP_APP_KEYS
    if key and key.strip()
]

if not TMAP_APP_KEYS:
    raise ValueError(
        ".env 파일에서 TMAP 앱키를 찾지 못했습니다.\n"
        "TMAP_APP_KEY_1부터 확인하세요."
    )

print(
    "로드된 TMAP 앱키 수:",
    len(TMAP_APP_KEYS),
)

# 보안을 위해 실제 키는 출력하지 않고 앞부분만 확인
for index, key in enumerate(
    TMAP_APP_KEYS,
    start=1,
):
    print(
        f"앱키 {index}:",
        f"{key[:4]}{'*' * 8}",
    )

로드된 TMAP 앱키 수: 5
앱키 1: ENz7********
앱키 2: amfF********
앱키 3: 0bf6********
앱키 4: x25N********
앱키 5: 26pn********


In [12]:
# =========================================================
# 15. OD 테이블에 거주동·근무동 주민센터 좌표 결합
# =========================================================

coord_table = (
    dong_geocoded[
        [
            "행정동 코드",
            "행정동 이름",
            "대표_장소명",
            "대표_주소",
            "대표_경도",
            "대표_위도",
            "지오코딩_상태",
        ]
    ]
    .copy()
)

# 거주동 좌표 테이블
home_coords = coord_table.rename(
    columns={
        "행정동 코드": "거주동 코드",
        "행정동 이름": "거주동_좌표기준_이름",
        "대표_장소명": "거주동_대표장소",
        "대표_주소": "거주동_대표주소",
        "대표_경도": "거주동_경도",
        "대표_위도": "거주동_위도",
        "지오코딩_상태": "거주동_좌표상태",
    }
)

# 근무동 좌표 테이블
work_coords = coord_table.rename(
    columns={
        "행정동 코드": "근무동 코드",
        "행정동 이름": "근무동_좌표기준_이름",
        "대표_장소명": "근무동_대표장소",
        "대표_주소": "근무동_대표주소",
        "대표_경도": "근무동_경도",
        "대표_위도": "근무동_위도",
        "지오코딩_상태": "근무동_좌표상태",
    }
)

api_targets_with_coords = (
    api_targets
    .merge(
        home_coords,
        on="거주동 코드",
        how="left",
        validate="many_to_one",
    )
    .merge(
        work_coords,
        on="근무동 코드",
        how="left",
        validate="many_to_one",
    )
)

print(
    "전체 API 대상 OD:",
    f"{len(api_targets_with_coords):,}",
)

print(
    "거주동 좌표 누락:",
    api_targets_with_coords[
        "거주동_경도"
    ].isna().sum(),
)

print(
    "근무동 좌표 누락:",
    api_targets_with_coords[
        "근무동_경도"
    ].isna().sum(),
)

display(
    api_targets_with_coords.head()
)

전체 API 대상 OD: 30,411
거주동 좌표 누락: 0
근무동 좌표 누락: 0


,OD_ID,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,...,거주동_대표주소,거주동_경도,거주동_위도,거주동_좌표상태,근무동_좌표기준_이름,근무동_대표장소,근무동_대표주소,근무동_경도,근무동_위도,근무동_좌표상태
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,55710.25,564412.04,0.098705,0.098705,...,서울 종로구 자하문로 92,126.97065,37.584116,성공,사직동,사직동주민센터,서울 종로구 경희궁1길 15,126.970591,37.571354,성공
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,47373.52,564412.04,0.083934,0.182639,...,서울 종로구 자하문로 92,126.97065,37.584116,성공,종로1.2.3.4가동,"종로1,2,3,4가동 주민센터",서울 종로구 삼일대로30길 47,126.990287,37.574436,성공
2,11110515_11140550,11110515,청운효자동,11140550,명동,4,21721.17,564412.04,0.038485,0.297715,...,서울 종로구 자하문로 92,126.97065,37.584116,성공,명동,명동주민센터 임시청사,서울 중구 퇴계로20나길 16,126.985242,37.558356,성공
3,11110515_11140520,11110515,청운효자동,11140520,소공동,5,17247.38,564412.04,0.030558,0.328273,...,서울 종로구 자하문로 92,126.97065,37.584116,성공,소공동,소공동주민센터,서울 중구 서소문로12길 36,126.974262,37.561670,성공
4,11110515_11560540,11110515,청운효자동,11560540,여의동,6,17222.83,564412.04,0.030515,0.358788,...,서울 종로구 자하문로 92,126.97065,37.584116,성공,여의동,여의동주민센터,서울 영등포구 국제금융로 124,126.934608,37.517604,성공


In [26]:
# =========================================================
# 16. TMAP 테스트 대상 OD 1개 선택
# =========================================================

required_coord_cols = [
    "거주동_경도",
    "거주동_위도",
    "근무동_경도",
    "근무동_위도",
]

tmap_test_candidates = (
    api_targets_with_coords[
        api_targets_with_coords[
            required_coord_cols
        ].notna().all(axis=1)
    ]
    .copy()
)

# 출발동과 도착동이 같은 내부 이동은 경로 테스트에서 제외
tmap_test_candidates = tmap_test_candidates[
    tmap_test_candidates["거주동 코드"]
    != tmap_test_candidates["근무동 코드"]
].copy()

if tmap_test_candidates.empty:
    raise ValueError(
        "TMAP 테스트가 가능한 외부 이동 OD가 없습니다."
    )

# 출근 이동량 컬럼이 있으면 이동량이 가장 큰 OD 선택
if "출근 이동량" in tmap_test_candidates.columns:
    tmap_test_candidates = (
        tmap_test_candidates
        .sort_values(
            "출근 이동량",
            ascending=False,
        )
    )

tmap_test_row = tmap_test_candidates.iloc[0]

print(
    "테스트 OD:",
    tmap_test_row["거주동 이름"],
    "→",
    tmap_test_row["근무동 이름"],
)

print(
    "출발 장소:",
    tmap_test_row["거주동_대표장소"],
)

print(
    "도착 장소:",
    tmap_test_row["근무동_대표장소"],
)

print(
    "출발 좌표:",
    tmap_test_row["거주동_경도"],
    tmap_test_row["거주동_위도"],
)

print(
    "도착 좌표:",
    tmap_test_row["근무동_경도"],
    tmap_test_row["근무동_위도"],
)

display(
    tmap_test_row.to_frame(
        name="값"
    )
)

테스트 OD: 청운효자동 → 사직동
출발 장소: 청운효자동주민센터
도착 장소: 사직동주민센터
출발 좌표: 126.97064969123 37.5841161738413
도착 좌표: 126.97059130939671 37.57135447739795


,값
OD_ID,11110515_11110530
거주동 코드,11110515
거주동 이름,청운효자동
근무동 코드,11110530
근무동 이름,사직동
목적지_순위,1
출근_이동량,55710.25
거주동_전체_출근량,564412.04
목적지_출근비중,0.098705
누적_출근비중,0.098705


In [27]:
# =========================================================
# 17. TMAP 대중교통 경로 조회 함수
# =========================================================

TMAP_TRANSIT_URL = (
    "https://apis.openapi.sk.com/transit/routes"
)


def request_tmap_transit_route(
    start_lon: float,
    start_lat: float,
    end_lon: float,
    end_lat: float,
    app_key: str,
    timeout: int = 30,
) -> dict:
    """
    TMAP 대중교통 API로 경로를 조회한다.

    주의
    ----
    경도: X
    위도: Y
    """

    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
        "appKey": app_key,
    }

    payload = {
        "startX": str(start_lon),
        "startY": str(start_lat),
        "endX": str(end_lon),
        "endY": str(end_lat),
        "lang": 0,
        "format": "json",
        "count": 10,
    }

    try:
        response = requests.post(
            TMAP_TRANSIT_URL,
            headers=headers,
            json=payload,
            timeout=timeout,
        )

        # 오류가 나더라도 응답 내용을 확인하기 위해 먼저 저장
        try:
            response_data = response.json()
        except ValueError:
            response_data = {
                "raw_text": response.text,
            }

        return {
            "success": response.ok,
            "status_code": response.status_code,
            "response": response_data,
            "error": None,
        }

    except requests.Timeout:
        return {
            "success": False,
            "status_code": None,
            "response": None,
            "error": "요청 시간 초과",
        }

    except requests.RequestException as error:
        return {
            "success": False,
            "status_code": None,
            "response": None,
            "error": str(error),
        }

In [28]:
# =========================================================
# 18. TMAP 대중교통 경로 1건 호출
# =========================================================

tmap_test_result = request_tmap_transit_route(
    start_lon=float(
        tmap_test_row["거주동_경도"]
    ),
    start_lat=float(
        tmap_test_row["거주동_위도"]
    ),
    end_lon=float(
        tmap_test_row["근무동_경도"]
    ),
    end_lat=float(
        tmap_test_row["근무동_위도"]
    ),
    app_key=TMAP_APP_KEYS[0],
)

print(
    "호출 성공:",
    tmap_test_result["success"],
)

print(
    "HTTP 상태 코드:",
    tmap_test_result["status_code"],
)

if tmap_test_result["error"]:
    print(
        "호출 오류:",
        tmap_test_result["error"],
    )

호출 성공: True
HTTP 상태 코드: 200


In [29]:
# =========================================================
# 19. TMAP 원본 응답 확인
# =========================================================

print(
    json.dumps(
        tmap_test_result["response"],
        ensure_ascii=False,
        indent=2,
    )
)

{
  "metaData": {
    "requestParameters": {
      "busCount": 4,
      "expressbusCount": 0,
      "subwayCount": 0,
      "airplaneCount": 0,
      "locale": "ko",
      "endY": "37.57135447739795",
      "endX": "126.97059130939671",
      "wideareaRouteCount": 0,
      "subwayBusCount": 0,
      "startY": "37.5841161738413",
      "startX": "126.97064969123",
      "ferryCount": 0,
      "trainCount": 0,
      "reqDttm": "20260730154332"
    },
    "plan": {
      "itineraries": [
        {
          "fare": {
            "regular": {
              "totalFare": 1500,
              "currency": {
                "symbol": "￦",
                "currency": "원",
                "currencyCode": "KRW"
              }
            }
          },
          "totalTime": 1015,
          "legs": [
            {
              "mode": "WALK",
              "sectionTime": 149,
              "distance": 128,
              "start": {
                "name": "출발지",
                "lon": 126.97064969

In [30]:
# =========================================================
# 20. TMAP 첫 번째 추천 경로 요약값 추출
# =========================================================

response_data = tmap_test_result["response"]

itineraries = (
    response_data
    .get("metaData", {})
    .get("plan", {})
    .get("itineraries", [])
)

if not itineraries:
    raise ValueError(
        "TMAP 응답에서 경로 정보를 찾지 못했습니다."
    )

route = itineraries[0]

fare_info = (
    route
    .get("fare", {})
    .get("regular", {})
)

tmap_route_summary = {
    "거주동 코드": tmap_test_row["거주동 코드"],
    "거주동 이름": tmap_test_row["거주동 이름"],
    "근무동 코드": tmap_test_row["근무동 코드"],
    "근무동 이름": tmap_test_row["근무동 이름"],

    # 전체 경로 요약
    "총 소요시간_초": route.get("totalTime"),
    "총 소요시간_분": (
        round(route.get("totalTime", 0) / 60, 1)
        if route.get("totalTime") is not None
        else None
    ),
    "총 이동거리_m": route.get("totalDistance"),
    "총 이동거리_km": (
        round(route.get("totalDistance", 0) / 1000, 2)
        if route.get("totalDistance") is not None
        else None
    ),

    # 요금
    "총 요금_원": fare_info.get("totalFare"),

    # 도보
    "총 도보시간_초": route.get("totalWalkTime"),
    "총 도보시간_분": (
        round(route.get("totalWalkTime", 0) / 60, 1)
        if route.get("totalWalkTime") is not None
        else None
    ),
    "총 도보거리_m": route.get("totalWalkDistance"),

    # 환승 및 교통수단
    "환승횟수": route.get("transferCount"),
    "보행구간수": route.get("pathType"),
}

tmap_route_summary_df = pd.DataFrame(
    [tmap_route_summary]
)

display(
    tmap_route_summary_df.T.rename(
        columns={0: "값"}
    )
)

,값
거주동 코드,11110515
거주동 이름,청운효자동
근무동 코드,11110530
근무동 이름,사직동
총 소요시간_초,1015
총 소요시간_분,16.9
총 이동거리_m,1465
총 이동거리_km,1.47
총 요금_원,1500
총 도보시간_초,887


In [31]:
# =========================================================
# 21. TMAP 경로 세부 구간 추출
# =========================================================

legs = route.get("legs", [])

leg_rows = []

for leg_index, leg in enumerate(
    legs,
    start=1,
):
    mode = leg.get("mode")

    leg_row = {
        "구간순서": leg_index,
        "교통수단": mode,
        "출발지": (
            leg.get("start", {})
            .get("name")
        ),
        "도착지": (
            leg.get("end", {})
            .get("name")
        ),
        "구간시간_초": leg.get("sectionTime"),
        "구간시간_분": (
            round(leg.get("sectionTime", 0) / 60, 1)
            if leg.get("sectionTime") is not None
            else None
        ),
        "구간거리_m": leg.get("distance"),
    }

    # 버스·지하철 등 대중교통 노선 정보
    route_info = leg.get("route")

    if isinstance(route_info, str):
        leg_row["노선명"] = route_info
    elif isinstance(route_info, dict):
        leg_row["노선명"] = (
            route_info.get("name")
            or route_info.get("routeName")
        )
    else:
        leg_row["노선명"] = None

    leg_rows.append(leg_row)

tmap_legs_df = pd.DataFrame(
    leg_rows
)

display(
    tmap_legs_df
)

,구간순서,교통수단,출발지,도착지,구간시간_초,구간시간_분,구간거리_m,노선명
0,1,WALK,출발지,신교동,149,2.5,128,NaN
1,2,BUS,신교동,경복궁역2번출구,128,2.1,715,지선:1711
2,3,WALK,경복궁역2번출구,도착지,738,12.3,848,NaN


In [32]:
# =========================================================
# 22. 교통수단별 이용 구간 수 계산
# =========================================================

mode_counts = (
    tmap_legs_df["교통수단"]
    .value_counts()
    .to_dict()
)

bus_count = mode_counts.get("BUS", 0)
subway_count = mode_counts.get("SUBWAY", 0)
walk_count = mode_counts.get("WALK", 0)

print("버스 이용 구간 수:", bus_count)
print("지하철 이용 구간 수:", subway_count)
print("도보 구간 수:", walk_count)

print(
    "전체 교통수단 순서:",
    " → ".join(
        tmap_legs_df["교통수단"]
        .fillna("UNKNOWN")
        .astype(str)
        .tolist()
    )
)

버스 이용 구간 수: 1
지하철 이용 구간 수: 0
도보 구간 수: 2
전체 교통수단 순서: WALK → BUS → WALK


In [33]:
# =========================================================
# 23. TMAP 테스트 1건 최종 결과
# =========================================================

tmap_route_summary_df[
    "버스_이용구간수"
] = bus_count

tmap_route_summary_df[
    "지하철_이용구간수"
] = subway_count

tmap_route_summary_df[
    "도보_구간수"
] = walk_count

tmap_route_summary_df[
    "교통수단_순서"
] = " → ".join(
    tmap_legs_df["교통수단"]
    .fillna("UNKNOWN")
    .astype(str)
    .tolist()
)

display(
    tmap_route_summary_df
)

,거주동 코드,거주동 이름,근무동 코드,근무동 이름,총 소요시간_초,총 소요시간_분,총 이동거리_m,총 이동거리_km,총 요금_원,총 도보시간_초,총 도보시간_분,총 도보거리_m,환승횟수,보행구간수,버스_이용구간수,지하철_이용구간수,도보_구간수,교통수단_순서
0,11110515,청운효자동,11110530,사직동,1015,16.9,1465,1.47,1500,887,14.8,976,0,2,1,0,2,WALK → BUS → WALK


In [35]:
# =========================================================
# 24. TMAP 전체 OD 호출 설정
# =========================================================

from pathlib import Path
from datetime import datetime
import time
import json
import pandas as pd
import requests


# 저장 폴더
TMAP_OUTPUT_DIR = Path("project_data/api_results")

TMAP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 최종 결과 파일
TMAP_RESULT_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_routes.csv"
)

# 실패 결과 파일
TMAP_FAILURE_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_failures.csv"
)


# 앱키별 하루 최대 호출 수
# 실제 콘솔 한도를 확인한 뒤 필요하면 수정
MAX_CALLS_PER_KEY = 1000

# 중간 저장 간격
SAVE_EVERY = 50

# 요청 사이 대기시간
REQUEST_INTERVAL = 0.2

print("TMAP 앱키 수:", len(TMAP_APP_KEYS))

print(
    "이번 실행 최대 호출 가능 수:",
    len(TMAP_APP_KEYS) * MAX_CALLS_PER_KEY,
)

print("결과 저장 경로:", TMAP_RESULT_FILE)

TMAP 앱키 수: 5
이번 실행 최대 호출 가능 수: 5000
결과 저장 경로: project_data/api_results/tmap_transit_routes.csv


In [36]:
# =========================================================
# 25. 전체 TMAP 호출 대상 정리
# =========================================================

required_coord_cols = [
    "거주동_경도",
    "거주동_위도",
    "근무동_경도",
    "근무동_위도",
]


# 좌표가 모두 존재하는 OD
tmap_targets = api_targets_with_coords[
    api_targets_with_coords[
        required_coord_cols
    ].notna().all(axis=1)
].copy()


# 같은 행정동 내부 이동 여부
tmap_targets["내부이동여부"] = (
    tmap_targets["거주동 코드"].astype(str)
    == tmap_targets["근무동 코드"].astype(str)
)


# 외부 이동만 API 호출
tmap_external_targets = (
    tmap_targets[
        ~tmap_targets["내부이동여부"]
    ]
    .copy()
)


# 이동량이 큰 OD부터 호출
if "출근 이동량" in tmap_external_targets.columns:
    tmap_external_targets = (
        tmap_external_targets
        .sort_values(
            "출근 이동량",
            ascending=False,
        )
        .reset_index(drop=True)
    )
else:
    tmap_external_targets = (
        tmap_external_targets
        .reset_index(drop=True)
    )


# OD 식별키 생성
tmap_external_targets["OD_KEY"] = (
    tmap_external_targets["거주동 코드"]
    .astype(str)
    .str.strip()
    + "_"
    + tmap_external_targets["근무동 코드"]
    .astype(str)
    .str.strip()
)


print(
    "좌표가 있는 전체 OD:",
    f"{len(tmap_targets):,}",
)

print(
    "내부 이동 OD:",
    f"{tmap_targets['내부이동여부'].sum():,}",
)

print(
    "TMAP 호출 대상 외부 이동 OD:",
    f"{len(tmap_external_targets):,}",
)

display(
    tmap_external_targets[
        [
            "OD_KEY",
            "거주동 이름",
            "근무동 이름",
            "거주동_대표장소",
            "근무동_대표장소",
        ]
        + (
            ["출근 이동량"]
            if "출근 이동량"
            in tmap_external_targets.columns
            else []
        )
    ].head()
)

좌표가 있는 전체 OD: 30,411
내부 이동 OD: 0
TMAP 호출 대상 외부 이동 OD: 30,411


,OD_KEY,거주동 이름,근무동 이름,거주동_대표장소,근무동_대표장소
0,11110515_11110530,청운효자동,사직동,청운효자동주민센터,사직동주민센터
1,11110515_11110615,청운효자동,종로1.2.3.4가동,청운효자동주민센터,"종로1,2,3,4가동 주민센터"
2,11110515_11140550,청운효자동,명동,청운효자동주민센터,명동주민센터 임시청사
3,11110515_11140520,청운효자동,소공동,청운효자동주민센터,소공동주민센터
4,11110515_11560540,청운효자동,여의동,청운효자동주민센터,여의동주민센터


In [37]:
# =========================================================
# 26. TMAP 응답 파싱 함수
# =========================================================

def parse_tmap_first_route(
    response_data: dict,
) -> dict:
    """
    TMAP 대중교통 응답에서
    첫 번째 추천 경로의 주요 정보를 추출한다.
    """

    itineraries = (
        response_data
        .get("metaData", {})
        .get("plan", {})
        .get("itineraries", [])
    )

    if not itineraries:
        raise ValueError(
            "응답에 itineraries 경로가 없습니다."
        )

    route = itineraries[0]

    regular_fare = (
        route
        .get("fare", {})
        .get("regular", {})
    )

    legs = route.get("legs", [])

    mode_list = []
    route_name_list = []

    bus_count = 0
    subway_count = 0
    walk_count = 0
    train_count = 0

    for leg in legs:
        mode = leg.get("mode")

        if mode:
            mode_list.append(str(mode))

        if mode == "BUS":
            bus_count += 1

        elif mode == "SUBWAY":
            subway_count += 1

        elif mode == "WALK":
            walk_count += 1

        elif mode == "TRAIN":
            train_count += 1

        route_info = leg.get("route")

        route_name = None

        if isinstance(route_info, str):
            route_name = route_info

        elif isinstance(route_info, dict):
            route_name = (
                route_info.get("name")
                or route_info.get("routeName")
            )

        if route_name:
            route_name_list.append(
                str(route_name)
            )

    total_time = route.get("totalTime")
    total_distance = route.get("totalDistance")
    total_walk_time = route.get("totalWalkTime")
    total_walk_distance = route.get(
        "totalWalkDistance"
    )

    return {
        "추천경로수": len(itineraries),
        "총소요시간_초": total_time,
        "총소요시간_분": (
            round(total_time / 60, 1)
            if total_time is not None
            else None
        ),
        "총이동거리_m": total_distance,
        "총이동거리_km": (
            round(total_distance / 1000, 2)
            if total_distance is not None
            else None
        ),
        "예상대중교통요금_원": (
            regular_fare.get("totalFare")
        ),
        "총도보시간_초": total_walk_time,
        "총도보시간_분": (
            round(total_walk_time / 60, 1)
            if total_walk_time is not None
            else None
        ),
        "총도보거리_m": total_walk_distance,
        "환승횟수": route.get(
            "transferCount"
        ),
        "버스_이용구간수": bus_count,
        "지하철_이용구간수": subway_count,
        "도보_구간수": walk_count,
        "기차_이용구간수": train_count,
        "교통수단_순서": (
            " → ".join(mode_list)
        ),
        "이용노선": (
            " → ".join(route_name_list)
        ),
    }

In [38]:
# =========================================================
# 27. TMAP 앱키 순환 호출 관리자
# =========================================================

class TmapKeyManager:
    def __init__(
        self,
        app_keys: list[str],
        max_calls_per_key: int = 1000,
    ):
        if not app_keys:
            raise ValueError(
                "사용 가능한 TMAP 앱키가 없습니다."
            )

        self.app_keys = app_keys
        self.max_calls_per_key = (
            max_calls_per_key
        )

        self.current_key_index = 0

        self.call_counts = {
            index: 0
            for index in range(
                len(app_keys)
            )
        }

        self.disabled_keys = set()

    def get_current_key(self):
        """
        호출 가능한 현재 앱키를 반환한다.
        모든 키가 소진되면 None을 반환한다.
        """

        for _ in range(
            len(self.app_keys)
        ):
            index = self.current_key_index

            count = self.call_counts[index]

            available = (
                index not in self.disabled_keys
                and count
                < self.max_calls_per_key
            )

            if available:
                return index, self.app_keys[index]

            self.current_key_index = (
                self.current_key_index + 1
            ) % len(self.app_keys)

        return None, None

    def record_call(
        self,
        key_index: int,
    ):
        self.call_counts[key_index] += 1

    def move_to_next_key(self):
        self.current_key_index = (
            self.current_key_index + 1
        ) % len(self.app_keys)

    def disable_key(
        self,
        key_index: int,
    ):
        self.disabled_keys.add(
            key_index
        )

        self.move_to_next_key()

    def print_status(self):
        print("\n앱키별 호출 현황")

        for index in range(
            len(self.app_keys)
        ):
            status = (
                "사용 중지"
                if index in self.disabled_keys
                else "사용 가능"
            )

            print(
                f"앱키 {index + 1}: "
                f"{self.call_counts[index]:,}"
                f"/{self.max_calls_per_key:,}건 "
                f"({status})"
            )

In [39]:
# =========================================================
# 28. TMAP OD 1건 호출 함수
# =========================================================

TMAP_TRANSIT_URL = (
    "https://apis.openapi.sk.com/transit/routes"
)


def call_tmap_route_once(
    row: pd.Series,
    app_key: str,
    timeout: int = 30,
) -> dict:
    """
    OD 한 건을 TMAP 대중교통 API에 요청한다.
    """

    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
        "appKey": app_key,
    }

    payload = {
        "startX": str(
            float(row["거주동_경도"])
        ),
        "startY": str(
            float(row["거주동_위도"])
        ),
        "endX": str(
            float(row["근무동_경도"])
        ),
        "endY": str(
            float(row["근무동_위도"])
        ),
        "lang": 0,
        "format": "json",
        "count": 10,
    }

    try:
        response = requests.post(
            TMAP_TRANSIT_URL,
            headers=headers,
            json=payload,
            timeout=timeout,
        )

        try:
            response_data = response.json()

        except ValueError:
            response_data = {
                "raw_text": response.text,
            }

        return {
            "success": response.ok,
            "status_code": response.status_code,
            "data": response_data,
            "error": None,
        }

    except requests.Timeout:
        return {
            "success": False,
            "status_code": None,
            "data": None,
            "error": "요청 시간 초과",
        }

    except requests.RequestException as error:
        return {
            "success": False,
            "status_code": None,
            "data": None,
            "error": str(error),
        }

In [40]:
# =========================================================
# 29. 기존 TMAP 결과 불러오기
# =========================================================

if TMAP_RESULT_FILE.exists():
    existing_results = pd.read_csv(
        TMAP_RESULT_FILE,
        encoding="utf-8-sig",
        dtype={
            "OD_KEY": "string",
            "거주동 코드": "string",
            "근무동 코드": "string",
        },
    )

    completed_od_keys = set(
        existing_results["OD_KEY"]
        .dropna()
        .astype(str)
    )

    print(
        "기존 성공 결과:",
        f"{len(existing_results):,}건",
    )

else:
    existing_results = pd.DataFrame()

    completed_od_keys = set()

    print("기존 성공 결과 없음")


remaining_targets = (
    tmap_external_targets[
        ~tmap_external_targets[
            "OD_KEY"
        ].isin(completed_od_keys)
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "남은 호출 대상:",
    f"{len(remaining_targets):,}건",
)

기존 성공 결과 없음
남은 호출 대상: 30,411건


In [44]:
# 현재 프로젝트 폴더 안 project_data 사용
PROJECT_DATA_DIR = Path.cwd() / "project_data"

TMAP_OUTPUT_DIR = PROJECT_DATA_DIR / "api_results"

TMAP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TMAP_RESULT_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_routes_detail_100.csv"
)

TMAP_FAILURE_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_failures_detail_100.csv"
)

print("현재 작업 폴더:", Path.cwd())
print("저장 폴더 존재 여부:", TMAP_OUTPUT_DIR.exists())
print("저장 폴더:", TMAP_OUTPUT_DIR.resolve())

현재 작업 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
저장 폴더 존재 여부: True
저장 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results


In [46]:
# =========================================================
# 30. 유료 TMAP 상세 대중교통 API 100건 이어서 호출
# =========================================================

from pathlib import Path
from datetime import datetime
import json
import time
import pandas as pd


# ---------------------------------------------------------
# 1. 상세 대중교통 API 주소 확인
# ---------------------------------------------------------

TMAP_TRANSIT_URL = (
    "https://apis.openapi.sk.com/transit/routes"
)

print(
    "호출 API:",
    TMAP_TRANSIT_URL,
)


# ---------------------------------------------------------
# 2. 유료 결제를 연결한 4번 앱키만 사용
# ---------------------------------------------------------

TMAP_PAID_APP_KEY = TMAP_APP_KEYS[3]

print(
    "사용 앱키:",
    "TMAP_APP_KEY_4",
)


# ---------------------------------------------------------
# 3. 저장 경로 설정 및 폴더 생성
# ---------------------------------------------------------

TMAP_OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / "api_results"
)

TMAP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TMAP_RESULT_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_routes_detail_100.csv"
)

TMAP_FAILURE_FILE = (
    TMAP_OUTPUT_DIR
    / "tmap_transit_failures_detail_100.csv"
)

print(
    "성공 결과 저장 경로:",
    TMAP_RESULT_FILE,
)

print(
    "실패 결과 저장 경로:",
    TMAP_FAILURE_FILE,
)


# ---------------------------------------------------------
# 4. 첫 100개 OD만 테스트 범위로 지정
# ---------------------------------------------------------

first_100_targets = (
    remaining_targets
    .head(100)
    .copy()
)


# ---------------------------------------------------------
# 5. 기존 성공 결과 불러오기
# ---------------------------------------------------------

if TMAP_RESULT_FILE.exists():

    existing_results = pd.read_csv(
        TMAP_RESULT_FILE,
        encoding="utf-8-sig",
        dtype={
            "OD_KEY": "string",
            "거주동 코드": "string",
            "근무동 코드": "string",
        },
    )

    saved_od_keys = set(
        existing_results["OD_KEY"]
        .dropna()
        .astype(str)
    )

else:

    existing_results = pd.DataFrame()

    saved_od_keys = set()


print(
    "기존 저장 성공:",
    f"{len(saved_od_keys):,}건",
)


# ---------------------------------------------------------
# 6. 첫 100건 중 이미 성공한 OD 제외
# ---------------------------------------------------------

test_targets = (
    first_100_targets[
        ~first_100_targets[
            "OD_KEY"
        ]
        .astype(str)
        .isin(saved_od_keys)
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "100건 중 이어 호출할 대상:",
    f"{len(test_targets):,}건",
)


# 이미 100건이 모두 완료된 경우
if test_targets.empty:

    print(
        "첫 100개 OD가 이미 모두 저장되어 있습니다."
    )


# ---------------------------------------------------------
# 7. 신규 성공·실패 결과 저장용 리스트
# ---------------------------------------------------------

new_result_rows = []
failure_rows = []

total_targets = len(
    test_targets
)


# ---------------------------------------------------------
# 8. 상세 대중교통 API 호출
# ---------------------------------------------------------

for row_index, row in (
    test_targets.iterrows()
):

    current_number = (
        len(saved_od_keys)
        + row_index
        + 1
    )

    od_key = str(
        row["OD_KEY"]
    )

    print(
        f"\r진행: {current_number:,}"
        f"/100 | "
        f"{row['거주동 이름']} "
        f"→ {row['근무동 이름']}",
        end="",
    )


    # 유료 4번 키로 상세 대중교통 API 호출
    result = call_tmap_route_once(
        row=row,
        app_key=TMAP_PAID_APP_KEY,
    )

    status_code = result[
        "status_code"
    ]


    # =====================================================
    # 호출 성공
    # =====================================================

    if result["success"]:

        try:

            # 첫 번째 추천 경로 주요 정보 추출
            parsed = parse_tmap_first_route(
                result["data"]
            )


            # API가 반환한 전체 추천 경로
            itineraries = (
                result["data"]
                .get("metaData", {})
                .get("plan", {})
                .get("itineraries", [])
            )


            result_row = {

                "OD_KEY": od_key,

                "거주동 코드": row[
                    "거주동 코드"
                ],

                "거주동 이름": row[
                    "거주동 이름"
                ],

                "근무동 코드": row[
                    "근무동 코드"
                ],

                "근무동 이름": row[
                    "근무동 이름"
                ],

                "거주동_대표장소": row.get(
                    "거주동_대표장소"
                ),

                "근무동_대표장소": row.get(
                    "근무동_대표장소"
                ),

                "거주동_경도": row[
                    "거주동_경도"
                ],

                "거주동_위도": row[
                    "거주동_위도"
                ],

                "근무동_경도": row[
                    "근무동_경도"
                ],

                "근무동_위도": row[
                    "근무동_위도"
                ],

                "출근 이동량": row.get(
                    "출근 이동량"
                ),

                # 시간·거리·요금·환승 등
                **parsed,

                # 상세 API가 반환한 전체 추천 경로 수
                "전체추천경로수": len(
                    itineraries
                ),

                "사용앱키": (
                    "TMAP_APP_KEY_4"
                ),

                "API종류": (
                    "대중교통 상세 경로"
                ),

                "HTTP상태코드": (
                    status_code
                ),

                "호출상태": (
                    "성공"
                ),

                # 모든 추천 경로, legs, 노선,
                # 정류장 및 도보 정보가 들어 있는 원본 응답
                "원본응답_JSON": json.dumps(
                    result["data"],
                    ensure_ascii=False,
                ),

                "호출시각": (
                    datetime.now()
                    .strftime(
                        "%Y-%m-%d %H:%M:%S"
                    )
                ),
            }


            new_result_rows.append(
                result_row
            )


        except Exception as error:

            failure_rows.append(
                {

                    "OD_KEY": od_key,

                    "거주동 코드": row[
                        "거주동 코드"
                    ],

                    "거주동 이름": row[
                        "거주동 이름"
                    ],

                    "근무동 코드": row[
                        "근무동 코드"
                    ],

                    "근무동 이름": row[
                        "근무동 이름"
                    ],

                    "사용앱키": (
                        "TMAP_APP_KEY_4"
                    ),

                    "HTTP상태코드": (
                        status_code
                    ),

                    "실패유형": (
                        "응답파싱실패"
                    ),

                    "오류내용": str(
                        error
                    ),

                    "원본응답_JSON": json.dumps(
                        result["data"],
                        ensure_ascii=False,
                    ),

                    "호출시각": (
                        datetime.now()
                        .strftime(
                            "%Y-%m-%d %H:%M:%S"
                        )
                    ),
                }
            )


    # =====================================================
    # API 호출 실패
    # =====================================================

    else:

        response_text = json.dumps(
            result["data"],
            ensure_ascii=False,
        )


        failure_rows.append(
            {

                "OD_KEY": od_key,

                "거주동 코드": row[
                    "거주동 코드"
                ],

                "거주동 이름": row[
                    "거주동 이름"
                ],

                "근무동 코드": row[
                    "근무동 코드"
                ],

                "근무동 이름": row[
                    "근무동 이름"
                ],

                "사용앱키": (
                    "TMAP_APP_KEY_4"
                ),

                "HTTP상태코드": (
                    status_code
                ),

                "실패유형": (
                    "API호출실패"
                ),

                "오류내용": (
                    result["error"]
                    or response_text
                ),

                "원본응답_JSON": (
                    response_text
                ),

                "호출시각": (
                    datetime.now()
                    .strftime(
                        "%Y-%m-%d %H:%M:%S"
                    )
                ),
            }
        )


        # 인증·권한·한도 문제면 즉시 중단
        if status_code in {
            401,
            403,
            429,
        }:

            print(
                f"\n호출 중단: "
                f"HTTP {status_code}"
            )

            break


    # =====================================================
    # 20건마다 중간 저장
    # =====================================================

    processed_count = (
        len(new_result_rows)
        + len(failure_rows)
    )


    if (
        processed_count > 0
        and processed_count % 20 == 0
    ):

        current_new_results = (
            pd.DataFrame(
                new_result_rows
            )
        )


        # 기존 성공 결과 + 이번 신규 성공 결과
        if not existing_results.empty:

            combined_results = pd.concat(
                [
                    existing_results,
                    current_new_results,
                ],
                ignore_index=True,
            )

        else:

            combined_results = (
                current_new_results.copy()
            )


        if not combined_results.empty:

            combined_results = (
                combined_results
                .drop_duplicates(
                    subset=["OD_KEY"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            combined_results.to_csv(
                TMAP_RESULT_FILE,
                index=False,
                encoding="utf-8-sig",
            )


        # 실패 결과 저장
        if failure_rows:

            pd.DataFrame(
                failure_rows
            ).to_csv(
                TMAP_FAILURE_FILE,
                index=False,
                encoding="utf-8-sig",
            )


        print(
            f"\n중간 저장 완료: "
            f"{processed_count:,}건 처리"
        )


    time.sleep(
        REQUEST_INTERVAL
    )


# ---------------------------------------------------------
# 9. 반복 종료 후 최종 저장
# ---------------------------------------------------------

new_results_df = pd.DataFrame(
    new_result_rows
)

failure_df = pd.DataFrame(
    failure_rows
)


if not existing_results.empty:

    final_tmap_results = pd.concat(
        [
            existing_results,
            new_results_df,
        ],
        ignore_index=True,
    )

else:

    final_tmap_results = (
        new_results_df.copy()
    )


if not final_tmap_results.empty:

    final_tmap_results = (
        final_tmap_results
        .drop_duplicates(
            subset=["OD_KEY"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    final_tmap_results.to_csv(
        TMAP_RESULT_FILE,
        index=False,
        encoding="utf-8-sig",
    )


if not failure_df.empty:

    failure_df.to_csv(
        TMAP_FAILURE_FILE,
        index=False,
        encoding="utf-8-sig",
    )


# ---------------------------------------------------------
# 10. 실행 결과 출력
# ---------------------------------------------------------

print(
    "\n\nTMAP 상세 경로 100건 호출 종료"
)

print(
    "이번 실행 성공:",
    f"{len(new_results_df):,}건",
)

print(
    "이번 실행 실패:",
    f"{len(failure_df):,}건",
)

print(
    "누적 성공 결과:",
    f"{len(final_tmap_results):,}건",
)

print(
    "성공 결과 저장:",
    TMAP_RESULT_FILE,
)

print(
    "실패 결과 저장:",
    TMAP_FAILURE_FILE,
)

호출 API: https://apis.openapi.sk.com/transit/routes
사용 앱키: TMAP_APP_KEY_4
성공 결과 저장 경로: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_routes_detail_100.csv
실패 결과 저장 경로: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_failures_detail_100.csv
기존 저장 성공: 20건
100건 중 이어 호출할 대상: 80건
진행: 40/100 | 청운효자동 → 문정2동3동동
중간 저장 완료: 20건 처리
진행: 60/100 | 청운효자동 → 성수1가2동
중간 저장 완료: 40건 처리
진행: 80/100 | 사직동 → 종로5.6가동.4가동
중간 저장 완료: 60건 처리
진행: 100/100 | 사직동 → 평창동3동
중간 저장 완료: 80건 처리


TMAP 상세 경로 100건 호출 종료
이번 실행 성공: 80건
이번 실행 실패: 0건
누적 성공 결과: 100건
성공 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_routes_detail_100.csv
실패 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_dat

In [45]:
# =========================================================
# 오류 발생 전 수집된 결과 즉시 저장
# =========================================================

current_results_df = pd.DataFrame(
    new_result_rows
)

current_failures_df = pd.DataFrame(
    failure_rows
)

if not current_results_df.empty:
    current_results_df.to_csv(
        TMAP_RESULT_FILE,
        index=False,
        encoding="utf-8-sig",
    )

if not current_failures_df.empty:
    current_failures_df.to_csv(
        TMAP_FAILURE_FILE,
        index=False,
        encoding="utf-8-sig",
    )

print(
    "현재까지 성공 결과 저장:",
    len(current_results_df),
)

print(
    "현재까지 실패 결과 저장:",
    len(current_failures_df),
)

현재까지 성공 결과 저장: 20
현재까지 실패 결과 저장: 0


In [47]:
# =========================================================
# 31. 상세 대중교통 API 100건 최종 저장
# =========================================================

new_results_df = pd.DataFrame(
    new_result_rows
)

failure_df = pd.DataFrame(
    failure_rows
)


if not existing_results.empty:

    final_tmap_results = pd.concat(
        [
            existing_results,
            new_results_df,
        ],
        ignore_index=True,
    )

else:

    final_tmap_results = (
        new_results_df.copy()
    )


if not final_tmap_results.empty:

    final_tmap_results = (
        final_tmap_results
        .drop_duplicates(
            subset=["OD_KEY"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    final_tmap_results.to_csv(
        TMAP_RESULT_FILE,
        index=False,
        encoding="utf-8-sig",
    )


if not failure_df.empty:

    failure_df.to_csv(
        TMAP_FAILURE_FILE,
        index=False,
        encoding="utf-8-sig",
    )


print(
    "누적 성공 결과:",
    f"{len(final_tmap_results):,}건",
)

print(
    "이번 실행 실패:",
    f"{len(failure_df):,}건",
)

print(
    "성공 결과 저장:",
    TMAP_RESULT_FILE,
)

print(
    "실패 결과 저장:",
    TMAP_FAILURE_FILE,
)

누적 성공 결과: 100건
이번 실행 실패: 0건
성공 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_routes_detail_100.csv
실패 결과 저장: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_failures_detail_100.csv


In [48]:
# =========================================================
# 32. 100건 호출 결과 확인
# =========================================================

check_columns = [
    "OD_KEY",
    "거주동 이름",
    "근무동 이름",
    "전체추천경로수",
    "총소요시간_분",
    "총이동거리_km",
    "예상대중교통요금_원",
    "총도보시간_분",
    "총도보거리_m",
    "환승횟수",
    "교통수단_순서",
    "이용노선",
    "HTTP상태코드",
]

display(
    new_results_df[
        [
            col
            for col in check_columns
            if col in new_results_df.columns
        ]
    ].head(100)
)

,OD_KEY,거주동 이름,근무동 이름,전체추천경로수,총소요시간_분,총이동거리_km,예상대중교통요금_원,총도보시간_분,총도보거리_m,환승횟수,교통수단_순서,이용노선,HTTP상태코드
0,11110515_11170685,청운효자동,한남동,10,27.9,9.48,1500,7.6,491,2,WALK → BUS → WALK → BUS → WALK → BUS → WALK,지선:1711 → 간선:401 → 간선:N31,200
1,11110515_11410520,청운효자동,천연동,10,15.3,4.05,1500,4.2,239,2,WALK → BUS → WALK → BUS → WALK → BUS → WALK,지선:1711 → 간선:704 → 간선:470,200
2,11110515_11680521,청운효자동,논현1동,10,49.4,14.86,1600,7.5,528,1,WALK → BUS → WALK → BUS → WALK,지선:1711 → 간선:401,200
3,11110515_11110580,청운효자동,교남동,3,14.3,2.60,1500,8.8,582,1,WALK → BUS → WALK → BUS → WALK,지선:1711 → 간선:171,200
4,11110515_11680545,청운효자동,압구정동,10,41.7,10.86,1600,9.0,629,1,WALK → BUS → WALK → BUS → WALK,지선:7212 → 간선:301,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,11110530_11650651,사직동,양재1동,10,58.1,14.48,1600,9.8,747,1,WALK → BUS → WALK → BUS → WALK,간선:470 → 간선:341,200
76,11110530_11440630,사직동,신수동,10,25.9,5.64,1500,10.2,775,1,WALK → BUS → WALK → BUS → WALK,간선:160 → 지선:7013A,200
77,11110530_11710620,사직동,가락본동,10,63.3,26.96,1750,16.2,1116,0,WALK → SUBWAY → WALK,수도권3호선,200
78,11110530_11110550,사직동,부암동,6,23.8,3.08,1500,16.2,1099,0,WALK → BUS → WALK,지선:1020,200


In [49]:
# 저장 상태 최종 확인

print("성공 파일 존재:", TMAP_RESULT_FILE.exists())
print("실패 파일 존재:", TMAP_FAILURE_FILE.exists())

if TMAP_RESULT_FILE.exists():
    saved_results = pd.read_csv(
        TMAP_RESULT_FILE,
        encoding="utf-8-sig",
        dtype={"OD_KEY": "string"},
    )

    print("저장된 성공 결과:", len(saved_results))
    print("OD 중복 수:", saved_results["OD_KEY"].duplicated().sum())
    print("저장 경로:", TMAP_RESULT_FILE.resolve())

if TMAP_FAILURE_FILE.exists():
    saved_failures = pd.read_csv(
        TMAP_FAILURE_FILE,
        encoding="utf-8-sig",
    )

    print("저장된 실패 결과:", len(saved_failures))

성공 파일 존재: True
실패 파일 존재: False
저장된 성공 결과: 100
OD 중복 수: 0
저장 경로: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks/project_data/api_results/tmap_transit_routes_detail_100.csv


-----------------------------

In [11]:
# =========================================================
# 기존 주민센터 좌표 결과 불러오기
# =========================================================

DONG_GEOCODED_FILE = (
    PROCESSED_DIR
    / "commute_dong_geocoded_kakao.csv"
)

if not DONG_GEOCODED_FILE.exists():
    raise FileNotFoundError(
        "주민센터 좌표 결과 파일이 없습니다.\n"
        f"확인 경로: {DONG_GEOCODED_FILE}"
    )

dong_geocoded = pd.read_csv(
    DONG_GEOCODED_FILE,
    encoding="utf-8-sig",
    dtype={
        "행정동 코드": "string",
        "행정동 이름": "string",
    },
)

print("좌표 결과 파일:", DONG_GEOCODED_FILE)
print("좌표 행정동 수:", f"{len(dong_geocoded):,}개")
print(
    "좌표 성공 수:",
    f"{(dong_geocoded['지오코딩_상태'] == '성공').sum():,}개",
)

display(dong_geocoded.head())

좌표 결과 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_dong_geocoded_kakao.csv
좌표 행정동 수: 428개
좌표 성공 수: 428개


,행정동 코드,자치구,행정동 이름,주민센터_검색어,검색결과수,지오코딩_API,대표_장소명,대표_주소,대표_경도,대표_위도,지오코딩_상태,지오코딩_오류
0,11110515,종로구,청운효자동,서울특별시 종로구 청운효자동 주민센터,4,Kakao Local,청운효자동주민센터,서울 종로구 자하문로 92,126.970650,37.584116,성공,NaN
1,11110530,종로구,사직동,서울특별시 종로구 사직동 주민센터,2,Kakao Local,사직동주민센터,서울 종로구 경희궁1길 15,126.970591,37.571354,성공,NaN
2,11110540,종로구,삼청동,서울특별시 종로구 삼청동 주민센터,2,Kakao Local,삼청동주민센터,서울 종로구 삼청로 107,126.981758,37.584998,성공,NaN
3,11110550,종로구,부암동,서울특별시 종로구 부암동 주민센터,4,Kakao Local,부암동주민센터,서울 종로구 창의문로 145,126.964061,37.592414,성공,NaN
4,11110560,종로구,평창동,서울특별시 종로구 평창동 주민센터,4,Kakao Local,평창동주민센터,서울 종로구 평창문화로 65,126.968358,37.606392,성공,NaN
